# Parte 1: Criptografia Simétrica

A empresa de saúde ocupacional envia informações diariamente para uma empresa parceira com a
qual já mantém uma comunicação estabelecida.
As duas empresas já possuem previamente a mesma chave secreta, armazenada de maneira
segura

In [7]:
from cryptography.fernet import Fernet

In [8]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding

In [11]:
dados_colaborador = "nome: João da Silva, cpf: 123.456.789-00, empresa: Empresa Mohammad's, exame: Apto para exercer suas atividades"


In [19]:
chave = Fernet.generate_key()
mensagem_original = dados_colaborador.encode()
print(f"Mensagem original: {mensagem_original.decode()}")

Mensagem original: nome: João da Silva, cpf: 123.456.789-00, empresa: Empresa Mohammad's, exame: Apto para exercer suas atividades


In [20]:
fernet = Fernet(chave)
mensagem_criptografada = fernet.encrypt(mensagem_original)
print(f"Mensagem criptografada (bytes): {mensagem_criptografada}\n")

Mensagem criptografada (bytes): b'gAAAAABqkON4_fpFdu6irqIzwvPE1kEPD-IDcDqgc0upqMK_UXw-GllMkJSJBw5l-6UsSRSHtCPClb_lUUuI3zD-VPIHJSZv7R1KGfLz2vv0zE14e1CifqVw8jHPHz_7xlS6L9bMKOma7wixlqUlEkHqVxzsIS1f9yLlonxqlsPSkuFxdOOfn9wGyFzM7yQB-WLP6fDwBKyz0prRg9j1LKdQ-BfZnJw5OVYGUCOyBThTmWQcrPExhc8='



In [21]:
mensagem_descriptografada = fernet.decrypt(mensagem_criptografada)
print(f"Mensagem descriptografada com sucesso: {mensagem_descriptografada.decode()}")

Mensagem descriptografada com sucesso: nome: João da Silva, cpf: 123.456.789-00, empresa: Empresa Mohammad's, exame: Apto para exercer suas atividades


# Parte 2: Criptografia Assimétrica

Agora, a empresa de saúde ocupacional precisa transmitir uma informação confidencial para uma
nova empresa parceira.
Nesse caso, as duas empresas não possuem previamente uma mesma chave secreta
compartilhada.
A empresa que receberá as informações possui seu próprio par de chaves: pública e privada.

In [4]:
chave_privada_mohammad = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
chave_publica_mohammad = chave_privada_mohammad.public_key()

In [12]:
print(f"Dados originais: {dados_colaborador}")

Dados originais: nome: João da Silva, cpf: 123.456.789-00, empresa: Empresa Mohammad's, exame: Apto para exercer suas atividades


In [13]:
dados_cifrados = chave_publica_mohammad.encrypt(dados_colaborador.encode(),
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [14]:
print(f"Dados cifrada: {dados_cifrados}")

Dados cifrada: b'\xc7\xdb%Y%\x80^T\x82\xce3\xa2\xd5\xe0U\x8b=\x19{\xab|\xb4.\xab\xc9<;\xf6J\xf3\xc4 {j\xd4l\x10\x1e\xacA\x83\xf14\xbba\xd0\xd5\x9e{\xd7\xbc\xe1U\x15\x12Nv\xf0\x1f\x08J\x86\xcac\xc1\\\x1d)V\x85\xa4f\xa6\x83!*"\x00\xbc\x00\x9a:\xeb\x93\xe2\xb4\xa9\xbc\xaex\xd4X\xc9\x1f\x1d\x83\xba\xb7\xda\x93P\xc4 \xff\xf1\xdd\xf3\x171\xcd\x83\xb2\x7f.\r\x1f\xe0\x82<%\xa5F\xe2\xe7\xb5\xc6\xee\x189\x93\x03\'\xcb\xb0b\x1eM\xf8S\xaaKj>Z\x066\x98E\x16\xadM!5\xc2\xfa\x7fJ\n\xcad\x1bu\x13\x94T\xec\x885\x997\x1e\xee\xe7\x0b\xee\xc5[,\x9c\xc2\xd6Kc\x81\x82;\xdeT\xe0\xc9w\xe6<\xd6\xc33?\x10h\x00\xf4\xabL%\xa6M\x96ph%\x84\x97^\x9e\xdcI\xbb\xd8\xd3\x17G\xe3\xa6\x91\x18\x8c{\x03Y\xee\x01\xc5\x98q\xb7m(S\x8bC\xa2OI\x88_b\n\x85\xb9\xedJ\xe8\n1\xa4\xb4'


In [15]:
dados_recuperados = chave_privada_mohammad.decrypt(
    dados_cifrados,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [16]:
print(f"Dados recuperado: {dados_recuperados.decode()}")

Dados recuperado: nome: João da Silva, cpf: 123.456.789-00, empresa: Empresa Mohammad's, exame: Apto para exercer suas atividades


# Parte 3: Comparação

Após executar as duas implementações, responda:
1. Qual é a principal diferença entre a criptografia simétrica e a criptografia assimétrica em relação
às chaves utilizadas?

A principal diferença está na quantidade e no compartilhamento dessas chaves. Na criptografia simétrica, utiliza-se apenas uma única chave secreta para trancar e destrancar a informação. Já na criptografia assimétrica, utiliza-se um par de chaves diferentes: uma chave pública, que qualquer um pode usar para trancar o dado, e uma chave privada, que fica guardada em segredo com o dono e é a única capaz de destrancar a informação.


2. Considerando os dois cenários apresentados, explique por que foram utilizadas abordagens
criptográficas diferentes.

As abordagens foram diferentes porque a relação de comunicação entre as empresas mudou em cada cenário:
Na Parte 1, usou-se a criptografia simétrica porque as duas empresas já se conheciam e já tinham uma chave secreta compartilhada e guardada em segurança.
Na Parte 2, usou-se a criptografia assimétrica porque a empresa de saúde precisava enviar dados para uma nova parceira com quem nunca tinha conversado antes, ou seja, elas não tinham uma senha em comum. A criptografia assimétrica resolveu isso permitindo que a empresa de saúde usasse a chave pública da nova parceira para trancar os dados, garantindo que apenas a dona da chave privada correspondente consiga ler a mensagem.